# Researcher 1: Training Notebook

이 노트북은 `train.csv`를 이용해 학생 성취도 회귀 모델을 학습하고, 최종 모델을 `model.pkl`로 저장하는 과정을 기록합니다.

## 1. 환경과 경로 설정

노트북을 `mission-result` 폴더에서 실행해도 되고, `researcher1` 폴더에서 실행해도 되도록 경로를 자동으로 맞춥니다.

In [1]:
from pathlib import Path
import sys

CWD = Path.cwd().resolve()
if (CWD / 'data' / 'train.csv').exists():
    RESEARCHER1_DIR = CWD
    PROJECT_ROOT = CWD.parent
else:
    PROJECT_ROOT = CWD
    RESEARCHER1_DIR = PROJECT_ROOT / 'researcher1'

SHARED_DIR = PROJECT_ROOT / 'shared'
TRAIN_PATH = RESEARCHER1_DIR / 'data' / 'train.csv'
TEST_PATH = RESEARCHER1_DIR / 'data' / 'test.csv'

sys.path.insert(0, str(RESEARCHER1_DIR))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TRAIN_PATH:', TRAIN_PATH)
print('TEST_PATH:', TEST_PATH)
print('SHARED_DIR:', SHARED_DIR)

PROJECT_ROOT: C:\Users\amy\Desktop\sprint\sprint_ai07\미션\미션15\4팀_김도민\mission-result
TRAIN_PATH: C:\Users\amy\Desktop\sprint\sprint_ai07\미션\미션15\4팀_김도민\mission-result\researcher1\data\train.csv
TEST_PATH: C:\Users\amy\Desktop\sprint\sprint_ai07\미션\미션15\4팀_김도민\mission-result\researcher1\data\test.csv
SHARED_DIR: C:\Users\amy\Desktop\sprint\sprint_ai07\미션\미션15\4팀_김도민\mission-result\shared


## 2. 데이터 불러오기

학습 데이터에는 목표변수 `Performance Index`가 있고, 테스트 데이터에는 예측해야 할 입력 변수만 있습니다.

In [2]:
import pandas as pd

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print('train shape:', train_df.shape)
print('test shape:', test_df.shape)
display(train_df.head())
display(test_df.head())

train shape: (7000, 6)
test shape: (3000, 5)


,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,6,73,No,7,2,58.0
1,1,89,Yes,7,2,64.0
2,3,97,Yes,8,0,75.0
3,8,70,No,5,5,59.0
4,7,94,Yes,7,4,86.0


,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced
0,7,99,Yes,9,1
1,8,51,Yes,7,2
2,8,91,No,4,5
3,5,79,No,7,8
4,2,72,No,4,3


## 3. 기본 정보 확인

컬럼 타입, 결측치, 기초 통계량을 확인해서 전처리 방향을 정합니다.

In [3]:
display(train_df.info())
display(train_df.isna().sum())
display(train_df.describe())
display(train_df['Extracurricular Activities'].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 6 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Hours Studied                     7000 non-null   int64  
 1   Previous Scores                   7000 non-null   int64  
 2   Extracurricular Activities        7000 non-null   object 
 3   Sleep Hours                       7000 non-null   int64  
 4   Sample Question Papers Practiced  7000 non-null   int64  
 5   Performance Index                 7000 non-null   float64
dtypes: float64(1), int64(4), object(1)
memory usage: 328.3+ KB


None

Hours Studied                       0
Previous Scores                     0
Extracurricular Activities          0
Sleep Hours                         0
Sample Question Papers Practiced    0
Performance Index                   0
dtype: int64

,Hours Studied,Previous Scores,Sleep Hours,Sample Question Papers Practiced,Performance Index
count,7000.000000,7000.000000,7000.000000,7000.000000,7000.000000
mean,4.950000,69.429714,6.530571,4.607429,55.095143
std,2.590621,17.289197,1.696144,2.863550,19.151574
min,1.000000,40.000000,4.000000,0.000000,10.000000
25%,3.000000,54.000000,5.000000,2.000000,40.000000
50%,5.000000,69.000000,7.000000,5.000000,55.000000
75%,7.000000,85.000000,8.000000,7.000000,70.000000
max,9.000000,99.000000,9.000000,9.000000,100.000000


Extracurricular Activities
No     3522
Yes    3478
Name: count, dtype: int64

## 4. 목표변수와 입력 변수 관계 확인

`Performance Index`와 각 숫자형 변수의 상관관계를 확인합니다. 이 결과는 보고서의 모델링 요약에 활용할 수 있습니다.

In [4]:
numeric_columns = [
    'Hours Studied',
    'Previous Scores',
    'Sleep Hours',
    'Sample Question Papers Practiced',
    'Performance Index',
]

correlation = train_df[numeric_columns].corr(numeric_only=True)['Performance Index'].sort_values(ascending=False)
display(correlation)

Performance Index                   1.000000
Previous Scores                     0.913866
Hours Studied                       0.374150
Sample Question Papers Practiced    0.049908
Sleep Hours                         0.049278
Name: Performance Index, dtype: float64

## 5. 전처리와 모델 파이프라인

숫자형 변수는 표준화하고, 범주형 변수 `Extracurricular Activities`는 원-핫 인코딩합니다. 전처리와 모델을 하나의 파이프라인으로 묶어 저장하면 연구자 2가 같은 전처리를 재현할 수 있습니다.

In [5]:
from train_model import FEATURE_COLUMNS, TARGET_COLUMN, build_model_pipeline

pipeline = build_model_pipeline()
print(pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                                  ['Hours Studied',
                                                   'Previous Scores',
                                                   'Sleep Hours',
                                                   'Sample Question Papers '
                                                   'Practiced']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Extracurricular '
                                                   'Activities'])])),
                ('model', Ridge())])


## 6. 검증 성능 확인

미션 요구사항에 맞춰 RMSE로 모델 성능을 평가합니다.

In [6]:
from sklearn.model_selection import train_test_split
from train_model import calculate_rmse

X_train, X_valid, y_train, y_valid = train_test_split(
    train_df[FEATURE_COLUMNS],
    train_df[TARGET_COLUMN],
    test_size=0.2,
    random_state=42,
)

pipeline.fit(X_train, y_train)
valid_pred = pipeline.predict(X_valid)
rmse = calculate_rmse(y_valid, valid_pred)
print(f'Validation RMSE: {rmse:.4f}')

Validation RMSE: 2.0105


## 7. 최종 모델 저장

검증이 끝난 뒤 전체 학습 데이터로 최종 모델을 다시 학습하고, `shared/model.pkl`과 `shared/test.csv`를 생성합니다.

In [7]:
from train_model import train_and_save_model

metrics = train_and_save_model(
    train_path=TRAIN_PATH,
    test_path=TEST_PATH,
    output_dir=SHARED_DIR,
    random_state=42,
)

metrics

{'rmse': 2.0105,
 'train_rows': 7000,
 'validation_rows': 1400,
 'test_rows': 3000,
 'model_path': 'C:\\Users\\amy\\Desktop\\sprint\\sprint_ai07\\미션\\미션15\\4팀_김도민\\mission-result\\shared\\model.pkl',
 'test_path': 'C:\\Users\\amy\\Desktop\\sprint\\sprint_ai07\\미션\\미션15\\4팀_김도민\\mission-result\\shared\\test.csv',
 'feature_columns': ['Hours Studied',
  'Previous Scores',
  'Extracurricular Activities',
  'Sleep Hours',
  'Sample Question Papers Practiced'],
 'target_column': 'Performance Index',
 'model': 'Ridge(alpha=1.0)',
 'random_state': 42}

## 8. 저장 결과 확인

연구자 2가 사용할 파일이 공유 폴더에 생성되었는지 확인합니다.

In [8]:
for path in sorted(SHARED_DIR.glob('*')):
    print(path.name, path.stat().st_size)

metrics.json 628
model.pkl 3542
result.csv 61257
test.csv 37572
